# Startup site

Workbook + bindings → static site with overview, input guide, and output catalog.


In [5]:
import shutil
import sys
from pathlib import Path
from typing import Annotated, Literal, get_args, get_origin

import pandas as pd
from excel_grapher.core.cell_types import Between, RealBetween, normalize_cell_type_env_key
from excel_grapher.exporter import to_semantic_viz_payload, write_semantic_viz_html
from excel_grapher.grapher import DynamicRefConfig, create_dependency_graph
from excel_grapher.series_bindings import (
    derive_input_series,
    derive_output_series,
    load_series_bindings,
)

repo = Path("..").resolve()
sys.path.insert(0, str(repo))
import workbook_config as wc

workbook = repo / "data" / "tiny-dsa.xlsx"
out = repo / "artifacts" / "startup-site"
download_name = workbook.name


ModuleNotFoundError: No module named 'pandas'

In [ ]:
graph = create_dependency_graph(
    workbook,
    list(wc.TARGETS),
    load_values=True,
    dynamic_refs=DynamicRefConfig.from_constraints(wc.CONSTRAINTS, {}),
    blank_ranges=wc.BLANK_RANGES,
)
bindings = load_series_bindings(wc.BINDINGS_PATH, validate=False)
manifest = {s["id"]: s for s in bindings["series"] if "input" in s}
constraints = {normalize_cell_type_env_key(k): v for k, v in wc.CONSTRAINTS.items()}


def describe(c):
    origin, args = get_origin(c), get_args(c)
    if origin is Literal:
        return type(args[0]).__name__, list(args)
    if origin is Annotated:
        base, *meta = args
        for m in meta:
            if isinstance(m, (Between, RealBetween)):
                return getattr(base, "__name__", str(base)), f"[{m.min}, {m.max}]"
        return getattr(base, "__name__", str(base)), None
    return str(c), None


rows = []
for series in derive_input_series(graph, bindings, workbook=workbook):
    meta = manifest[series["id"]]
    c = constraints.get(normalize_cell_type_env_key(series["cells"][0]["address"]))
    dtype, acceptable = (
        describe(c) if c is not None else (meta["structure"]["measure"]["dtype"], None)
    )
    dims = [d["id"] for d in meta["structure"].get("dimensions", [])]
    rows.append(
        {
            "id": series["id"],
            "range": meta.get("data_range"),
            "dtype": dtype,
            "dimensions": dims or None,
            "acceptable": acceptable,
            "notes": meta.get("notes"),
        }
    )

guide = pd.DataFrame(rows)
guide


In [ ]:
output_manifest = {s["id"]: s for s in bindings["series"] if "output" in s}

output_rows = []
for series in derive_output_series(graph, bindings, workbook=workbook):
    meta = output_manifest[series["id"]]
    attrs = {
        a["concept"]: a.get("value") for a in meta["structure"].get("attributes", [])
    }
    dims = [d["id"] for d in meta["structure"].get("dimensions", [])]
    output_rows.append(
        {
            "id": series["id"],
            "range": meta.get("data_range"),
            "dtype": meta["structure"]["measure"]["dtype"],
            "dimensions": dims or None,
            "unit": attrs.get("UNIT_MEASURE"),
            "notes": meta.get("notes"),
        }
    )

outputs = pd.DataFrame(output_rows)
out.mkdir(parents=True, exist_ok=True)
guide.to_csv(out / "startup-guide.csv", index=False)
outputs.to_csv(out / "output-catalog.csv", index=False)
outputs


In [ ]:
payload = to_semantic_viz_payload(
    graph,
    bindings,
    workbook=workbook,
    blank_ranges=wc.BLANK_RANGES,
)
statement_graph_name = "statement-graph.html"
write_semantic_viz_html(
    payload,
    out / statement_graph_name,
    title=f"{wc.DIST_METADATA.library_name} statement graph",
)
stats = payload.graph.stats
print(
    f"wrote {out / statement_graph_name} "
    f"statements={stats.statement_count} "
    f"bundles={stats.bundle_count} "
    f"instance_edges={stats.instance_edge_count} "
    f"cells={stats.cell_count}"
)


In [ ]:
title = wc.DIST_METADATA.library_name
description = wc.DIST_METADATA.description

(out / "download").mkdir(exist_ok=True)
shutil.copy2(workbook, out / "download" / download_name)

style = """
    body { font-family: system-ui, sans-serif; max-width: 960px; margin: 2rem auto; padding: 0 1rem; }
    table { border-collapse: collapse; width: 100%; font-size: 0.9rem; }
    th, td { border: 1px solid #ccc; padding: 0.4rem 0.6rem; text-align: left; vertical-align: top; }
    th { background: #f4f4f4; }
    a.button { display: inline-block; margin: 1rem 0; padding: 0.6rem 1rem;
               background: #1a5f4a; color: #fff; text-decoration: none; border-radius: 4px; }
    nav a { margin-right: 1rem; }
    .diagram { width: 100%; height: 70vh; border: 1px solid #ccc; border-radius: 4px; }
"""

nav = """
  <nav>
    <a href="index.html">Overview</a>
    <a href="inputs.html">Inputs</a>
    <a href="outputs.html">Outputs</a>
  </nav>
"""

guide_html = guide.to_html(index=False, escape=True)
outputs_table_html = outputs.to_html(index=False, escape=True)

index_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>{title} — Overview</title>
  <style>{style}</style>
</head>
<body>
  {nav}
  <h1>{title}</h1>
  <p>{description}</p>
  <a class="button" href="download/{download_name}" download>Download spreadsheet</a>
  <h2>Statement graph</h2>
  <iframe class="diagram" src="{statement_graph_name}" title="Statement dependency graph"></iframe>
  <p><a href="{statement_graph_name}">Open full-page diagram</a></p>
</body>
</html>
"""

inputs_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>{title} — Inputs</title>
  <style>{style}</style>
</head>
<body>
  {nav}
  <h1>{title}</h1>
  <p>Edit only the input cells listed below. Leave formulas alone.</p>
  <a class="button" href="download/{download_name}" download>Download spreadsheet</a>
  <h2>Startup guide</h2>
  {guide_html}
</body>
</html>
"""

outputs_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8"/>
  <title>{title} — Outputs</title>
  <style>{style}</style>
</head>
<body>
  {nav}
  <h1>{title}</h1>
  <h2>Output catalog</h2>
  <p>Read-only results. Do not edit these cells in the spreadsheet.</p>
  {outputs_table_html}
</body>
</html>
"""

(out / "index.html").write_text(index_html, encoding="utf-8")
(out / "inputs.html").write_text(inputs_html, encoding="utf-8")
(out / "outputs.html").write_text(outputs_html, encoding="utf-8")
print(out)
print(f"Serve: uv run python -m http.server 8000 --directory {out}")
